# Assignment 3 -- Transformer

## Goal

1. Learning to design a Transformer architecture.

2. Learning to implement and realize single (and multi-head) attention mechanism.

3. Learning to implement the position-wise feedforward (FFN) Layer.


## Score (100% + 20% bonus)

1. Part 5: Position-wise feedforward (FFN) Layer 20%

2. Part 6: Single-head attention 20%
3. Part 6 Bonus: multi-head attention 20%

4. Part 7: Transformer architecture 20%

5. Model size 15%:

* 5%: If your model size is smaller than **1MB**, you will get 5%. Otherwise, no points will be awarded.
* 10%:  The remaining 10% will depend on your ranking within the class.

6. Model accuracy 15%:

  * 5%: If your accuracy is higher than **57%**, you will get 5%. Otherwise, no points will be awarded.
  * 10%:  The remaining 10% will depend on your ranking within the class.

7. Model accuracy on another dataset 10%: it will depend on your ranking within the class.


## Rule

1. You can use all torch and einops functions for your implementations
2. You can modify hyperparameters in part 9
3. DO NOT use  torch.nn.MultiheadAttention()
4. DO NOT attempt to modify the sections `DO NOT MODIFY`.
5. You may only modify the following parts:

- Part 5: `myFFN`
- Part 6: `myAttention`
- Part 7: `myTransformer`
- Part 9: model hyperparameters

## Submission

Upload your zip files to NTU Cool.
* StudentID_HW3.zip
  * DL_HW3_StudentID.ipynb
  * StudentID_submission.pt
  * StudentID_submission.csv

Deadline: 5/25 Mon. 11:59 AM

Please fill your student ID number below

In [1]:
# Please fill your student ID number
student_id = 'r14725055'

assert student_id != 'xxxxx', "Please fill in your student ID."

## Part 1

Import necessary library

`DO NOT MODIFY`

In [2]:
# If einops is not installed, run:
# !pip install einops

In [3]:
# Model
import torch
import torch.nn as nn
import torch.nn.functional as F

# Dataset
from torchvision import datasets
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
from scipy.io import loadmat

# Optimizer
from torch.optim.optimizer import Optimizer

# Pre-processing
import torchvision.transforms as trns
from PIL import Image

from math import sqrt

# For Transformer
# you may need using `pip install einops` to install einops
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

## Part 2

`DO NOT MODIFY`

Global variables.

If following code used these variables, please keep them when you modify the code

In [4]:
batch_size = 16
num_classes = 10
input_size = (32, 32, 3)
patch_size = 4
num_epoch = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part 3

`DO NOT MODIFY`

Create dataloader with pre-processing of dataset

In [5]:
# Create train/test transforms
train_transform = trns.Compose([
    trns.ToTensor(),
])

test_transform = trns.Compose([
    trns.ToTensor(),
])

# Create train/test datasets with pre-processing
# The dataset will automatic download if does not exist
data_train = datasets.CIFAR10(root='./dataset/', train=True, transform=train_transform, download=True)
data_test = datasets.CIFAR10(root='./dataset/', train=False, transform=test_transform, download=True)

# Create train/test dataloader for datasets with  pre-processing
train_loader = DataLoader(data_train, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(data_test,  batch_size=batch_size, shuffle=False)

100.0%
/Users/nalada16/Desktop/obs/ntu1-2/DL/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


## Part 4

`DO NOT MODIFY`

Convert image to token

In [6]:
class myTokenization(nn.Module):

    def __init__(self, output_dim, patch_size, channels):

        super().__init__()

        patch_dim = patch_size * patch_size * channels

        self.to_patch_tokens = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = patch_size, p2 = patch_size),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, output_dim)
        )

    def forward(self, x):

        return self.to_patch_tokens(x)


## Part 5

Please implement the following formula for the position-wise feedforward layer (FFN).

$FFN(x) = max(0, x W_1 + b_1)W_2 + b_2$

In [7]:
class myFFN(nn.Module):

    def __init__(self, input_dim, hidden_dim):
        super().__init__()

        # your implementation
        # example: single linear layer
        self.ffn = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),  
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim) 
        )
    def forward(self, x):

        # your implementation
        # example: single linear layer
        out = self.ffn(x)

        return out

## Part 6

Please implement scaled dot-product attention. You may implement single-head attention, but multi-head attention is encouraged.

$Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

_Bonus: please implement multi-head attention:_

$MultiHead(Q, K, V) = Concat(head_1, ..., head_h)W^O$, where $head_i = Attention(QW^Q_i, KW^K_i, VW^V_i)$

In [8]:
# class myAttention(nn.Module):

#     def __init__(self, input_dim, heads, head_dim):

#         super().__init__()

#         # your implementation

#     def forward(self, x):

#         # your implementation
#         # Please implement scaled dot-product attention here (bonus: multi-head attention)

#         return out

In [9]:
class myAttention(nn.Module):
    def __init__(self, input_dim, heads, head_dim):
        super().__init__()
        self.heads = heads
        self.head_dim = head_dim
        inner_dim = heads * head_dim                 # 所有 head 合併的維度
        self.scale = head_dim ** -0.5

        # 一次投影所有 head
        self.to_qkv = nn.Linear(input_dim, inner_dim * 3, bias=False)
        self.to_out = nn.Linear(inner_dim, input_dim)

    def forward(self, x):
        b, n, _ = x.shape
        h = self.heads

        # 一次算出 Q, K, V 並切分成 3 份
        qkv = self.to_qkv(x)
        qkv = qkv.chunk(3, dim=-1)
        q, k, v = map(
            lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv
        )

        # Scaled dot-product attention
        dots = torch.einsum('bhid,bhjd->bhij', q, k) * self.scale  # (B, h, N, N)
        attn = dots.softmax(dim=-1)

        # 加權求和
        out = torch.einsum('bhij,bhjd->bhid', attn, v)   # (B, h, N, head_dim)
        out = rearrange(out, 'b h n d -> b n (h d)')      # (B, N, h * head_dim)

        return self.to_out(out)                      # (B, N, input_dim)

## Part 7

Please design your Transformer architecture with your implementation of {myFFN, myAttention}.

You could decide number of layers, number of hidden neurons of each layer, activation function of each layer to design your Transformer.

Please notice that your score will depend on both size and accuracy of your model.

Here are the architectures of Transformer and ViT for your reference.

### Transformer

In [10]:
class myTransformer(nn.Module):

    def __init__(self, dim, heads, dim_head, mlp_dim):

        super().__init__()

        # your implementation
        num_layers = 6
        self.layers = nn.ModuleList([])
        for _ in range(num_layers):
            self.layers.append(nn.ModuleList([
                nn.LayerNorm(dim),
                myAttention(dim, heads, dim_head),
                nn.Dropout(0.1),
                nn.LayerNorm(dim),
                myFFN(dim, mlp_dim),
                nn.Dropout(0.1),
            ]))

        self.norm = nn.LayerNorm(dim)  # Final LayerNorm

    def forward(self, x):

        # your implementation
        for norm1, attn, drop1, norm2, ffn, drop2 in self.layers:
            x = drop1(attn(norm1(x))) + x
            x = drop2(ffn(norm2(x))) + x
        
        out = self.norm(x) 
        return out

## Part 8

`DO NOT MODIFY`

Construct Vision Transformer by implemented Transformer and FFN

![image.png](attachment:image.png)

In [11]:
class myViT(nn.Module):

    def __init__(self, input_size, patch_size, hidden_dim, heads, head_dim, mlp_dim, num_classes):

        super().__init__()

        image_height, image_width, channels = input_size
        patch_height, patch_width = patch_size, patch_size

        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = patch_height * patch_width * channels

        self.to_input_token = myTokenization(hidden_dim, patch_size, channels)
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches+1, hidden_dim))

        self.transformer = myTransformer(hidden_dim, heads, head_dim, mlp_dim)

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_classes))

    def forward(self, x):

        # convert image to token embeddings
        x = self.to_input_token(x)

        # concatenate cls token and input token and add positional embedding
        b, n, _ = x.shape
        cls_tokens = repeat(self.cls_token, '() n d -> b n d', b = b)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embedding[:, :(n + 1)]

        # transformer
        x = self.transformer(x)

        # cls token for prediction
        x = x[:, 0]

        return self.mlp_head(x)



## Part 9

You could adjust `hidden_dim`, `heads`, `head_dim`, and `mlp_dim` to construct your ViT.

In [12]:
model = myViT(
    input_size = input_size,
    patch_size = patch_size,
    hidden_dim = 64,
    heads = 4,
    head_dim = 64,
    mlp_dim = 64,
    num_classes = num_classes).to(device)


## Part 10

`DO NOT MODIFY`

Multiclass cross-entropy loss

In [13]:
criterion = nn.CrossEntropyLoss()

## Part 11

`DO NOT MODIFY`

Adam optimizer.

In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

## Part 12

`DO NOT MODIFY`

Model training

In [15]:
model.train()

for epoch in range(num_epoch):

    losses = []

    for batch_num, input_data in enumerate(train_loader):

        optimizer.zero_grad()

        x, y = input_data
        x = x.to(device).float()
        y = y.to(device)

        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        losses.append(loss.item())

        optimizer.step()

        if batch_num % 500 == 0:
            print('\tEpoch %d | Batch %d | Loss %6.4f' % (epoch, batch_num, loss.item()))

    print('Epoch %d | Loss %6.4f' % (epoch, sum(losses)/len(losses)))

torch.save(model, student_id + '_submission.pt')

	Epoch 0 | Batch 0 | Loss 2.6585
	Epoch 0 | Batch 500 | Loss 1.7495
	Epoch 0 | Batch 1000 | Loss 1.4766
	Epoch 0 | Batch 1500 | Loss 2.0085
	Epoch 0 | Batch 2000 | Loss 1.3880
	Epoch 0 | Batch 2500 | Loss 1.5830
	Epoch 0 | Batch 3000 | Loss 1.3364
Epoch 0 | Loss 1.6284
	Epoch 1 | Batch 0 | Loss 1.8633
	Epoch 1 | Batch 500 | Loss 1.6224
	Epoch 1 | Batch 1000 | Loss 1.3983
	Epoch 1 | Batch 1500 | Loss 1.4641
	Epoch 1 | Batch 2000 | Loss 1.7097
	Epoch 1 | Batch 2500 | Loss 1.5466
	Epoch 1 | Batch 3000 | Loss 1.9576
Epoch 1 | Loss 1.3240
	Epoch 2 | Batch 0 | Loss 0.5862
	Epoch 2 | Batch 500 | Loss 1.2923
	Epoch 2 | Batch 1000 | Loss 1.0762
	Epoch 2 | Batch 1500 | Loss 0.9990
	Epoch 2 | Batch 2000 | Loss 1.2998
	Epoch 2 | Batch 2500 | Loss 0.8806
	Epoch 2 | Batch 3000 | Loss 1.3296
Epoch 2 | Loss 1.2092
	Epoch 3 | Batch 0 | Loss 1.0727
	Epoch 3 | Batch 500 | Loss 1.0838
	Epoch 3 | Batch 1000 | Loss 1.0553
	Epoch 3 | Batch 1500 | Loss 0.7314
	Epoch 3 | Batch 2000 | Loss 0.9042
	Epoch 3 | Bat

## Part 13

`DO NOT MODIFY`

Model evaluation

In [16]:
import csv
model.eval()

with open(student_id + '_submission.csv', 'w') as f:

    fieldnames = ['ImageId', 'Prediction']

    writer = csv.DictWriter(f, fieldnames=fieldnames, lineterminator = '\n')
    writer.writeheader()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, t in test_loader:

            x = x.to(device).float()
            output = model(x).argmax(dim=1)

            for y,l in zip(output, t):

                writer.writerow({fieldnames[0]: (total+1),
                                 fieldnames[1]: y.item()})

                total += 1
                if y.item() == l.item():
                    correct += 1

    print('Accuracy: %6.4f' % (correct / total))

Accuracy: 0.6080
